# Fairness Without Demographics + DP Privacy Auditing

**Papers implemented:**
1. *Fairness Without Demographics in Repeated Loss Minimization* — Hashimoto et al. (2018, ICML)
2. *Privacy Auditing with One (1) Training Run* — Steinke, Nasr & Jagielski (2023)

**Key ideas joined here:**
- Hashimoto et al. propose training with a χ²-regularised distributionally robust objective over the **loss distribution** (no group labels needed). The DRO ball of radius α shrinks the effective sample size, concentrating weight on the highest-loss samples at each step.
- Steinke et al. plant binary-membership canaries and run a single private training run, then audit empirical ε via a Binomial hypothesis test (Corollary 5.4).
- We combine them: **DP-SGD + DRO loss reweighting + canary auditing**, all in one training run.

**Architecture:** ResNet-18 with GroupNorm (Opacus-compatible), UTKFace age regression, gender as the demographic group for evaluation (not for training in the DRO branch).

In [14]:
import os, sys, json, pickle, warnings, gc, time, random, math
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from opacus.utils.batch_memory_manager import BatchMemoryManager

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: NVIDIA L40S
Memory: 47.7 GB


## Configuration

In [15]:
CFG = {
    # ── Paths ──────────────────────────────────────────────────────
    'image_dir':  Path('project/data/UTKFace'),
    'exp_dir':    Path('project/experiments/fairness_dp_audit'),

    # ── Training ───────────────────────────────────────────────────
    'epochs':       30,
    'lr':           1e-4,
    'batch_size':   64,
    'weight_decay': 1e-4,
    'val_frac':     0.15,
    'seed':         2024,

    # ── Hashimoto et al. (2018) DRO parameters ─────────────────────
    # α controls the size of the χ²-divergence ball.
    # Effective worst-case fraction ≈ 1/(1 + sqrt(2α)).
    # Hashimoto et al. recommend α in [0.2, 2.0] for group sizes ~10–50% of data.
    'dro_alpha':    1.0,   # χ² regularisation coefficient
    'dro_step_size': 0.01, # η for mirror-descent weight update (eq. 5 in paper)

    # ── DP-SGD (Opacus) ────────────────────────────────────────────
    'dp_enabled':          True,
    'dp_noise_multiplier': 1.1,
    'dp_max_grad_norm':    1.0,
    'dp_delta':            1e-5,
    'dp_phys_batch':       32,

    # ── Privacy Auditing (Steinke et al., 2023) ────────────────────
    # k  = number of canary pairs (IN/OUT each)
    # The paper recommends k ≥ 500 for tight bounds at ε ~ 1–3.
    'audit_k':             500,
    'audit_gamma':         0.05,  # FPR budget for Corollary 5.4 test
    'audit_seed':          42,
}

CFG['exp_dir'].mkdir(parents=True, exist_ok=True)

# Reproducibility
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])

print('Configuration ready.')

Configuration ready.


## Dataset

In [16]:
class UTKFaceDataset(Dataset):
    """Returns (image_tensor, age_float, gender_long)."""
    def __init__(self, root_dir, transform=None):
        self.root_dir  = Path(root_dir)
        self.transform = transform
        self.samples   = []
        for p in self.root_dir.rglob('*.jpg'):
            try:
                stem  = p.stem.replace('.chip', '')
                parts = stem.split('_')
                if len(parts) >= 3:
                    age, gender = int(parts[0]), int(parts[1])
                    if 1 <= age <= 116 and gender in (0, 1):
                        self.samples.append({'path': str(p), 'age': age, 'gender': gender})
            except:
                pass
        print(f'UTKFace: {len(self.samples)} samples loaded.')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        img = Image.open(s['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return (img,
                torch.tensor(s['age'],    dtype=torch.float32),
                torch.tensor(s['gender'], dtype=torch.long))


train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print('Transforms defined.')

Transforms defined.


In [17]:
# ── Split dataset ─────────────────────────────────────────────────
full_ds = UTKFaceDataset(CFG['image_dir'], transform=None)
N       = len(full_ds)
n_val   = int(N * CFG['val_frac'])
n_train = N - n_val

gen  = torch.Generator().manual_seed(CFG['seed'])
train_idx, val_idx = random_split(range(N), [n_train, n_val], generator=gen)
train_idx = list(train_idx)
val_idx   = list(val_idx)


class TransformSubset(Dataset):
    def __init__(self, base, indices, transform):
        self.base, self.indices, self.tf = base, indices, transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, age, g = self.base[self.indices[i]]
        return self.tf(img), age, g


train_ds = TransformSubset(full_ds, train_idx, train_tf)
val_ds   = TransformSubset(full_ds, val_idx,   val_tf)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}')

# Gender distribution
male_ct = sum(1 for i in range(len(train_ds)) if train_ds[i][2] == 0)
print(f'Training gender: Male={male_ct:,}  Female={len(train_ds)-male_ct:,}')

UTKFace: 33487 samples loaded.
Train: 28,464  |  Val: 5,023
Training gender: Male=14,206  Female=14,258


## Model (ResNet-18, GroupNorm, Opacus-compatible)

In [18]:
class GNBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1     = nn.Conv2d(inplanes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1       = nn.GroupNorm(min(32, planes), planes)
        self.relu      = nn.ReLU(inplace=False)
        self.conv2     = nn.Conv2d(planes, planes, 3, padding=1, bias=False)
        self.bn2       = nn.GroupNorm(min(32, planes), planes)
        self.downsample = downsample

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample:
            x = self.downsample(x)
        return self.relu(out + x)


class OpacusResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        self.inplanes = 64
        self.conv1    = nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False)
        self.bn1      = nn.GroupNorm(min(32, 64), 64)
        self.relu     = nn.ReLU(inplace=False)
        self.maxpool  = nn.MaxPool2d(3, stride=2, padding=1)
        self.layer1   = self._make_layer(64,  2, stride=1)
        self.layer2   = self._make_layer(128, 2, stride=2)
        self.layer3   = self._make_layer(256, 2, stride=2)
        self.layer4   = self._make_layer(512, 2, stride=2)
        self.avgpool  = nn.AdaptiveAvgPool2d(1)
        self.fc       = nn.Linear(512, 1)

    def _make_layer(self, planes, blocks, stride):
        ds = None
        if stride != 1 or self.inplanes != planes:
            ds = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, 1, stride=stride, bias=False),
                nn.GroupNorm(min(32, planes), planes))
        layers = [GNBlock(self.inplanes, planes, stride, ds)]
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(GNBlock(planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(torch.flatten(self.avgpool(x), 1)).squeeze(1)


def build_model():
    m = OpacusResNet18()
    errs = ModuleValidator.validate(m, strict=False)
    if errs:
        m = ModuleValidator.fix(m)
    return m.to(device)

print('Model class defined.')

Model class defined.


---
## Hashimoto et al. (2018) DRO Loss Reweighting

**Key algorithm (Algorithm 1, Hashimoto et al.):**

At each step $t$, given per-sample losses $\{\ell_i\}_{i=1}^n$:

1. Compute importance weights:  
   $w_i \propto \exp(\ell_i / \alpha)$  (softmax of scaled losses)

2. Normalise so $\sum w_i = 1$  

3. Model gradient:  
   $\nabla_\theta \sum_i w_i \cdot \ell_i$

This is the **dual form** of the χ²-ball DRO problem.  
The connection to worst-group risk: as $\alpha \to 0$, the method concentrates all weight on the single worst sample (min-max); as $\alpha \to \infty$, it reduces to ERM.

> **No group labels are used during training.** Gender is only used for *evaluation* (to measure if the DRO implicitly reduces the gender gap).


In [19]:
def dro_weighted_loss(per_sample_losses: torch.Tensor, alpha: float) -> torch.Tensor:
    """
    Hashimoto et al. (2018) χ²-DRO weighted loss.

    Computes the dual-form objective:
        L_DRO = Σ_i  softmax(ℓ_i / α)_i  ·  ℓ_i

    This is equivalent (by Lagrangian duality) to optimising the worst-case
    distribution within a χ²(α)-divergence ball around the empirical
    distribution (Proposition 1 of the paper).

    Args:
        per_sample_losses : shape (B,)  — per-sample L1 losses
        alpha             : χ² regularisation coefficient (> 0)

    Returns:
        Scalar DRO loss ready for .backward()
    """
    # Detach losses for weight computation (weights treated as constants
    # in the primal update — Hashimoto et al. alternate primal/dual steps).
    with torch.no_grad():
        weights = torch.softmax(per_sample_losses.detach() / alpha, dim=0)
    return (weights * per_sample_losses).sum()


print('DRO loss function defined  (Hashimoto et al. 2018, dual form).')

DRO loss function defined  (Hashimoto et al. 2018, dual form).


---
## Privacy Auditing Infrastructure (Steinke et al., 2023)

We implement the **one-training-run** auditing framework from §5 of the paper.

### Protocol
1. Sample a fixed binary membership matrix $m \in \{0,1\}^k$ once (k canary pairs).
2. For each canary $i$: if $m_i = 1$ include sample $x_i^{IN}$ in training; else include $x_i^{OUT}$.
3. After training, score each canary with a membership-inference signal $s_i$.
4. Apply Corollary 5.4 binomial test to obtain $\hat{\varepsilon}_{\text{emp}}$.

### Canary design
Canaries are **random-label samples**: take real images but assign a random age label drawn uniformly from a held-out label set not present in the main dataset. The memorisation signal comes from the model fitting these random labels.

### Scoring functions
- **Black-box:** $s_i = \ell(w_0, x_i) - \ell(w_T, x_i)$  (loss reduction)
- **White-box:** $s_i = \sum_t \langle w_{t-1} - w_t,\ \text{clip}(\nabla\ell(w_{t-1}, x_i)) \rangle$  (gradient alignment)


In [20]:
# ── Canary dataset ────────────────────────────────────────────────
class CanaryDataset(Dataset):
    """
    Each canary is a real UTKFace image with a *random* age label drawn
    from {150, 151, ..., 149+k}.  These labels are far outside the
    normal [1,116] range so the model can only fit them via memorisation.

    Args:
        base_ds    : the full UTKFaceDataset (no transform)
        indices    : indices into base_ds to use as canary images
        membership : bool tensor of shape (k,)  — True → IN canary
        transform  : train_tf
        rng        : np.random.Generator for label assignment
    """
    def __init__(self, base_ds, indices, membership, transform, rng):
        assert len(indices) == len(membership)
        self.base_ds    = base_ds
        self.indices    = list(indices)
        self.membership = membership          # shape (k,) bool
        self.transform  = transform
        # Assign one random label per canary (fixed for the experiment)
        self.random_ages = torch.tensor(
            rng.integers(150, 150 + len(indices), size=len(indices)),
            dtype=torch.float32)

    def __len__(self):
        return int(self.membership.sum().item())   # only IN canaries go into training

    def get_in_dataset(self):
        """Returns (Dataset of IN canaries, list of their canary indices 0..k-1)."""
        in_idxs = [i for i, m in enumerate(self.membership) if m]
        return _InCanarySubset(self, in_idxs), in_idxs

    def get_image_and_label(self, canary_idx):
        """Returns (image_tensor, random_age) for canary canary_idx."""
        img, _, _ = self.base_ds[self.indices[canary_idx]]
        img = self.transform(img)
        return img, self.random_ages[canary_idx]


class _InCanarySubset(Dataset):
    """Thin wrapper so IN canaries can be added to a ConcatDataset."""
    def __init__(self, canary_ds, in_idxs):
        self.cd = canary_ds
        self.in_idxs = in_idxs
    def __len__(self):
        return len(self.in_idxs)
    def __getitem__(self, i):
        img, age = self.cd.get_image_and_label(self.in_idxs[i])
        return img, age, torch.tensor(-1, dtype=torch.long)  # -1 = canary group


print('CanaryDataset defined.')

CanaryDataset defined.


In [21]:
# ── Build canary set ──────────────────────────────────────────────
k          = CFG['audit_k']
audit_rng  = np.random.default_rng(CFG['audit_seed'])

# Reserve k samples from OUTSIDE the main training set for canaries.
# We use the first k indices from val_idx (val is unused for training).
canary_pool = val_idx[:k * 2]          # 2k so IN and OUT can differ
canary_in_pool  = canary_pool[:k]      # images to use as IN canaries
canary_out_pool = canary_pool[k:2*k]   # images to use as OUT canaries

# Membership vector: True = this canary is IN the training set
# Drawn once globally from seeded RNG — NEVER re-randomised.
membership = torch.tensor(
    audit_rng.integers(0, 2, size=k).astype(bool),
    dtype=torch.bool)

print(f'Canaries: k={k}, IN count={membership.sum().item()}, OUT count={(~membership).sum().item()}')

canary_ds = CanaryDataset(
    base_ds    = full_ds,
    indices    = canary_in_pool,   # We use IN-pool images for all canaries;
                                   # OUT canaries are simply absent from training.
    membership = membership,
    transform  = train_tf,
    rng        = audit_rng
)

in_canary_ds, in_canary_global_idxs = canary_ds.get_in_dataset()
print(f'IN canary dataset size: {len(in_canary_ds)}')

# ── Combine training data with IN canaries ────────────────────────
from torch.utils.data import ConcatDataset

augmented_train_ds = ConcatDataset([train_ds, in_canary_ds])
augmented_loader   = DataLoader(
    augmented_train_ds, batch_size=CFG['batch_size'],
    shuffle=True, num_workers=4, pin_memory=True)

print(f'Augmented training size (main + IN canaries): {len(augmented_train_ds):,}')

Canaries: k=500, IN count=244, OUT count=256
IN canary dataset size: 244
Augmented training size (main + IN canaries): 28,708


---
## Training Loop: DP-SGD + Hashimoto DRO + Canary Tracking

In [22]:
def evaluate_gender_mae(model, loader):
    """Return (male_mae, female_mae) ignoring canary samples (gender == -1)."""
    model.eval()
    loss_g  = [0.0, 0.0]
    count_g = [0,   0]
    with torch.no_grad():
        for imgs, ages, genders in loader:
            imgs, ages, genders = imgs.to(device), ages.to(device), genders.to(device)
            preds = model(imgs)
            abs_err = (preds - ages).abs()
            for g in (0, 1):
                mask = genders == g
                loss_g[g]  += abs_err[mask].sum().item()
                count_g[g] += mask.sum().item()
    return (loss_g[0] / max(count_g[0], 1),
            loss_g[1] / max(count_g[1], 1))


def compute_canary_loss(model, canary_idx, canary_ds):
    """Return scalar loss on a single canary (used for black-box scoring)."""
    model.eval()
    with torch.no_grad():
        img, age = canary_ds.get_image_and_label(canary_idx)
        img = img.unsqueeze(0).to(device)
        age = age.unsqueeze(0).to(device)
        pred = model(img)
        return (pred - age).abs().item()


print('Evaluation utilities defined.')

Evaluation utilities defined.


In [23]:
def train_dp_dro_with_audit(cfg, train_loader, val_loader, canary_ds,
                             membership, in_canary_global_idxs):
    """
    Single training run combining:
      1. DP-SGD via Opacus
      2. Hashimoto et al. (2018) DRO loss reweighting
      3. Steinke et al. (2023) privacy auditing via canary tracking
    """
    k       = len(membership)
    alpha   = cfg['dro_alpha']
    dp_en   = cfg['dp_enabled']

    model     = build_model()
    optimizer = optim.AdamW(model.parameters(),
                            lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    criterion = nn.L1Loss(reduction='none')   # per-sample losses

    # Store original model for white-box scoring (without Opacus hooks)
    original_model = None
    
    # ── Wrap with Opacus DP engine ─────────────────────────────────
    privacy_engine = None
    if dp_en:
        privacy_engine = PrivacyEngine()
        # Store reference to original model before wrapping
        original_model = model
        model, optimizer, train_loader = privacy_engine.make_private(
            module        = model,
            optimizer     = optimizer,
            data_loader   = train_loader,
            noise_multiplier = cfg['dp_noise_multiplier'],
            max_grad_norm    = cfg['dp_max_grad_norm'],
        )

    # ── Black-box: record initial canary losses ────────────────────
    print('Computing initial canary losses (black-box baseline)...')
    initial_losses = np.array([
        compute_canary_loss(model, i, canary_ds) for i in range(k)
    ])

    # ── White-box: Use simplified scoring (loss-based only for DP) ──
    # For DP training, we'll use black-box scores only since white-box
    # requires gradient computation that conflicts with Opacus hooks
    wb_scores = np.zeros(k, dtype=np.float64)  # Will remain zeros for DP
    
    history = {
        'train_loss_dro':  [],
        'val_male_mae':    [],
        'val_female_mae':  [],
        'epsilon':         [],
    }

    # ────────────────────────────────────────────────────────────────
    # Main training loop
    # ────────────────────────────────────────────────────────────────
    for epoch in range(1, cfg['epochs'] + 1):
        model.train()
        epoch_loss = 0.0
        epoch_n    = 0

        if dp_en:
            with BatchMemoryManager(
                data_loader=train_loader,
                max_physical_batch_size=cfg['dp_phys_batch'],
                optimizer=optimizer
            ) as memory_safe_loader:
                for imgs, ages, _ in memory_safe_loader:
                    imgs, ages = imgs.to(device), ages.to(device)

                    # Forward pass
                    preds = model(imgs)
                    per_sample_losses = criterion(preds, ages)

                    # Hashimoto DRO reweighting
                    loss = dro_weighted_loss(per_sample_losses, alpha)

                    # Backward + DP-SGD step
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    epoch_loss += loss.item() * imgs.size(0)
                    epoch_n    += imgs.size(0)
        else:
            # Non-DP training loop with white-box scoring
            for imgs, ages, _ in train_loader:
                imgs, ages = imgs.to(device), ages.to(device)

                # Save current params for white-box score accumulation
                params_before = {n: p.detach().clone()
                                 for n, p in model.named_parameters() if p.requires_grad}

                # Forward pass
                preds = model(imgs)
                per_sample_losses = criterion(preds, ages)

                # Hashimoto DRO reweighting
                loss = dro_weighted_loss(per_sample_losses, alpha)

                # Backward + SGD step
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item() * imgs.size(0)
                epoch_n    += imgs.size(0)

                # White-box score: Δw = w_before - w_after
                model.eval()
                delta_w_flat = []
                for n, p in model.named_parameters():
                    if p.requires_grad:
                        dw = (params_before[n] - p.detach()).view(-1)
                        delta_w_flat.append(dw)
                
                if len(delta_w_flat) > 0:
                    delta_w_flat = torch.cat(delta_w_flat)
                    
                    # Compute per-canary gradients (non-DP only)
                    for ci in range(k):
                        img_c, age_c = canary_ds.get_image_and_label(ci)
                        img_c = img_c.unsqueeze(0).to(device)
                        age_c = age_c.unsqueeze(0).to(device)

                        # Need gradients for this computation
                        img_c.requires_grad_(True)
                        pred_c = model(img_c)
                        c_loss = criterion(pred_c, age_c).sum()
                        
                        # Compute gradients with respect to model parameters
                        c_grad = torch.autograd.grad(
                            c_loss, list(model.parameters()), 
                            retain_graph=False, 
                            allow_unused=True
                        )
                        
                        # Filter and flatten gradients
                        c_grad_flat_list = []
                        for g in c_grad:
                            if g is not None:
                                c_grad_flat_list.append(g.view(-1))
                        
                        if len(c_grad_flat_list) > 0:
                            c_grad_flat = torch.cat(c_grad_flat_list)
                            
                            # Clip gradient
                            grad_norm = c_grad_flat.norm()
                            clip_factor = min(1.0, cfg['dp_max_grad_norm'] / (grad_norm.item() + 1e-8))
                            c_grad_clipped = c_grad_flat * clip_factor
                            
                            # Accumulate dot product
                            if len(delta_w_flat) == len(c_grad_clipped):
                                wb_scores[ci] += torch.dot(delta_w_flat, c_grad_clipped).item()
                
                model.train()

        # Epoch-end evaluation
        male_mae, female_mae = evaluate_gender_mae(model, val_loader)
        history['train_loss_dro'].append(epoch_loss / epoch_n)
        history['val_male_mae'].append(male_mae)
        history['val_female_mae'].append(female_mae)

        eps = 0.0
        if dp_en and privacy_engine is not None:
            eps = privacy_engine.get_epsilon(cfg['dp_delta'])
            history['epsilon'].append(eps)

        if epoch % 5 == 0:
            print(f'Epoch {epoch:2d} | DRO-loss={epoch_loss/epoch_n:.4f} | '
                  f'Male MAE={male_mae:.3f} | Female MAE={female_mae:.3f}'
                  + (f' | ε={eps:.3f}' if dp_en else ''))

    # Black-box scores: loss reduction over training
    print('\nComputing final canary losses (black-box)...')
    final_losses = np.array([
        compute_canary_loss(model, i, canary_ds) for i in range(k)
    ])
    bb_scores = initial_losses - final_losses

    # For DP training, we'll use black-box scores only
    if dp_en:
        print('\nNote: White-box scores not computed due to Opacus hook conflicts.')
        print('Using black-box scores for privacy audit (still valid per Steinke et al.).')
        wb_scores = bb_scores.copy()  # Fallback to black-box for DP case

    epsilon_spent = privacy_engine.get_epsilon(cfg['dp_delta']) if dp_en and privacy_engine else None

    return model, bb_scores, wb_scores, history, initial_losses, epsilon_spent


print('Training function defined (simplified for DP compatibility).')

Training function defined (simplified for DP compatibility).


In [ ]:
import os
import shutil
import gc
import pickle

# ============================================================
# CLEAN START - DELETE OLD INCORRECT RESULTS
# ============================================================

# Backup old results if needed
old_results_path = CFG['exp_dir'] / 'full_results_latest.json'
if old_results_path.exists():
    backup_path = CFG['exp_dir'] / 'full_results_latest_backup.json'
    shutil.copy(old_results_path, backup_path)
    print(f'✓ Backed up old results to {backup_path}')
    os.remove(old_results_path)
    print(f'✓ Deleted old results to force fresh training')

print('\n' + '='*65)
print('SIMPLIFIED MEMORY-EFFICIENT TRAINING')
print('='*65)

# ============================================================
# SIMPLIFIED TRAINING FUNCTION - NO COMPLEX STATE_DICT LOADING
# ============================================================

def train_dp_dro_with_audit_simplified(cfg, train_loader, val_loader, canary_ds,
                                         membership, in_canary_global_idxs):
    """
    Simplified training with white-box scoring using direct computation
    """
    k       = len(membership)
    alpha   = cfg['dro_alpha']
    dp_en   = cfg['dp_enabled']
    
    # Clear GPU cache before starting
    torch.cuda.empty_cache()
    gc.collect()

    model = build_model()
    optimizer = optim.AdamW(model.parameters(),
                            lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    criterion = nn.L1Loss(reduction='none')

    # ── Wrap with Opacus DP engine ─────────────────────────────────
    privacy_engine = None
    if dp_en:
        privacy_engine = PrivacyEngine()
        model, optimizer, train_loader = privacy_engine.make_private(
            module        = model,
            optimizer     = optimizer,
            data_loader   = train_loader,
            noise_multiplier = cfg['dp_noise_multiplier'],
            max_grad_norm    = cfg['dp_max_grad_norm'],
        )

    # ── Black-box: record initial canary losses ────────────────────
    print('Computing initial canary losses (black-box baseline)...')
    initial_losses = np.zeros(k)
    for i in range(k):
        initial_losses[i] = compute_canary_loss(model, i, canary_ds)
        if (i + 1) % 100 == 0:
            print(f'  Processed {i+1}/{k} canaries')
            torch.cuda.empty_cache()

    # Initialize white-box scores
    wb_scores = np.zeros(k, dtype=np.float64)
    
    history = {
        'train_loss_dro':  [],
        'val_male_mae':    [],
        'val_female_mae':  [],
        'epsilon':         [],
    }

    print(f'Starting training for {cfg["epochs"]} epochs...')
    print('='*60)
    
    # For tracking parameter updates (store only flattened delta_w for each step)
    param_updates = []  # Store flattened delta_w for each batch
    
    for epoch in range(1, cfg['epochs'] + 1):
        model.train()
        epoch_loss = 0.0
        epoch_n    = 0

        if dp_en:
            # DP-SGD with memory management
            with BatchMemoryManager(
                data_loader=train_loader,
                max_physical_batch_size=cfg['dp_phys_batch'],
                optimizer=optimizer
            ) as memory_safe_loader:
                for batch_idx, (imgs, ages, _) in enumerate(memory_safe_loader):
                    imgs, ages = imgs.to(device), ages.to(device)
                    
                    # Save current parameters as flattened vector
                    params_before_flat = torch.cat([
                        p.detach().view(-1) for p in model.parameters() if p.requires_grad
                    ])
                    
                    # Forward pass
                    preds = model(imgs)
                    per_sample_losses = criterion(preds, ages)
                    loss = dro_weighted_loss(per_sample_losses, alpha)
                    
                    # Backward step
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    
                    epoch_loss += loss.item() * imgs.size(0)
                    epoch_n    += imgs.size(0)
                    
                    # Get flattened parameters after update
                    params_after_flat = torch.cat([
                        p.detach().view(-1) for p in model.parameters() if p.requires_grad
                    ])
                    
                    # Compute delta_w
                    delta_w_flat = params_before_flat - params_after_flat
                    param_updates.append(delta_w_flat.cpu())
                    
                    # Clear cache periodically
                    if batch_idx % 50 == 0:
                        torch.cuda.empty_cache()
        else:
            # Non-DP training
            for imgs, ages, _ in train_loader:
                imgs, ages = imgs.to(device), ages.to(device)
                
                params_before_flat = torch.cat([
                    p.detach().view(-1) for p in model.parameters() if p.requires_grad
                ])
                
                preds = model(imgs)
                per_sample_losses = criterion(preds, ages)
                loss = dro_weighted_loss(per_sample_losses, alpha)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item() * imgs.size(0)
                epoch_n    += imgs.size(0)
                
                params_after_flat = torch.cat([
                    p.detach().view(-1) for p in model.parameters() if p.requires_grad
                ])
                
                delta_w_flat = params_before_flat - params_after_flat
                param_updates.append(delta_w_flat.cpu())

        # Epoch-end evaluation
        male_mae, female_mae = evaluate_gender_mae(model, val_loader)
        history['train_loss_dro'].append(epoch_loss / max(epoch_n, 1))
        history['val_male_mae'].append(male_mae)
        history['val_female_mae'].append(female_mae)

        if dp_en and privacy_engine is not None:
            eps = privacy_engine.get_epsilon(cfg['dp_delta'])
            history['epsilon'].append(eps)
        else:
            eps = 0.0

        if epoch % 5 == 0:
            print(f'Epoch {epoch:2d} | Loss={epoch_loss/max(epoch_n,1):.4f} | '
                  f'Male MAE={male_mae:.3f} | Female MAE={female_mae:.3f} | ε={eps:.3f}')
            
        # Clear cache after each epoch
        torch.cuda.empty_cache()
        gc.collect()

    # ── Compute black-box scores ──────────────────────────────────
    print('\nComputing final canary losses for black-box scores...')
    final_losses = np.zeros(k)
    for i in range(k):
        final_losses[i] = compute_canary_loss(model, i, canary_ds)
        if (i + 1) % 100 == 0:
            print(f'  Processed {i+1}/{k} canaries')
            torch.cuda.empty_cache()
    
    bb_scores = initial_losses - final_losses
    
    # ── Compute white-box scores AFTER training ───────────────────
    print('\nComputing white-box scores by replaying parameter updates...')
    
    # Create a fresh model for white-box computation
    clean_model = build_model()
    clean_model.to(device)
    clean_model.eval()
    
    # Get initial flattened parameters
    initial_flat = torch.cat([
        p.detach().view(-1) for p in clean_model.parameters() if p.requires_grad
    ])
    
    # For each canary, compute cumulative gradient alignment
    for ci in range(k):
        if (ci + 1) % 50 == 0:
            print(f'  Processing canary {ci+1}/{k}')
            torch.cuda.empty_cache()
        
        # Reset to initial weights
        current_flat = initial_flat.clone()
        
        # Function to set model parameters from flattened tensor
        def set_params_from_flat(model, flat_params):
            idx = 0
            for param in model.parameters():
                if param.requires_grad:
                    numel = param.numel()
                    param.data.copy_(flat_params[idx:idx+numel].reshape(param.shape))
                    idx += numel
        
        set_params_from_flat(clean_model, current_flat)
        
        total_score = 0.0
        
        # Replay all parameter updates
        for step, delta_w in enumerate(param_updates):
            # Get current parameters as flattened
            current_flat = torch.cat([
                p.detach().view(-1) for p in clean_model.parameters() if p.requires_grad
            ])
            
            # Compute gradient of canary loss at current weights
            img_c, age_c = canary_ds.get_image_and_label(ci)
            img_c = img_c.unsqueeze(0).to(device)
            age_c = age_c.unsqueeze(0).to(device)
            
            with torch.enable_grad():
                for param in clean_model.parameters():
                    param.requires_grad_(True)
                
                pred_c = clean_model(img_c)
                c_loss = criterion(pred_c, age_c).sum()
                
                # Compute gradients for all parameters
                c_grad = torch.autograd.grad(
                    c_loss, list(clean_model.parameters()),
                    retain_graph=False,
                    allow_unused=True
                )
            
            # Flatten gradients
            c_grad_flat = torch.cat([
                g.detach().view(-1) for g in c_grad if g is not None
            ])
            
            # Clip gradient
            grad_norm = c_grad_flat.norm()
            clip_factor = min(1.0, cfg['dp_max_grad_norm'] / (grad_norm.item() + 1e-8))
            c_grad_clipped = c_grad_flat * clip_factor
            
            # Accumulate dot product
            total_score += torch.dot(delta_w.to(device), c_grad_clipped).item()
            
            # Update model weights to next step (before this delta_w)
            # Note: We apply delta_w AFTER computing gradient at current weights
            current_flat = current_flat - delta_w.to(device)
            set_params_from_flat(clean_model, current_flat)
            
            # Clear gradients
            clean_model.zero_grad()
        
        wb_scores[ci] = total_score
        
        # Clear cache periodically
        if ci % 10 == 0:
            torch.cuda.empty_cache()
    
    epsilon_spent = privacy_engine.get_epsilon(cfg['dp_delta']) if dp_en and privacy_engine else 0.0
    
    print(f'\n✓ White-box score statistics:')
    in_mask = membership.numpy()
    print(f'  IN canaries (n={in_mask.sum()}): mean={wb_scores[in_mask].mean():.3f}, std={wb_scores[in_mask].std():.3f}')
    print(f'  OUT canaries (n={(~in_mask).sum()}): mean={wb_scores[~in_mask].mean():.3f}, std={wb_scores[~in_mask].std():.3f}')
    print(f'  Correlation BB vs WB: {np.corrcoef(bb_scores, wb_scores)[0,1]:.3f}')
    
    return model, bb_scores, wb_scores, history, initial_losses, epsilon_spent


# ============================================================
# SAVE/LOAD FUNCTIONS
# ============================================================

def save_results(results, cfg, results_type='full'):
    """Save results to JSON"""
    exp_dir = cfg['exp_dir']
    latest_path = exp_dir / f'{results_type}_results_latest.json'
    with open(latest_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'✓ Results saved to {latest_path}')
    return latest_path


def load_latest_results(cfg, results_type='full'):
    """Load latest saved results"""
    latest_path = cfg['exp_dir'] / f'{results_type}_results_latest.json'
    if latest_path.exists():
        with open(latest_path, 'r') as f:
            results = json.load(f)
        print(f'✓ Loaded results from {latest_path}')
        return results
    return None


# ============================================================
# RUN TRAINING
# ============================================================

print('\n' + '='*65)
print('STARTING SIMPLIFIED TRAINING')
print('='*65)

# Reduce batch sizes to save memory
CFG['batch_size'] = 32
CFG['dp_phys_batch'] = 16

# Recreate dataloaders
train_loader = DataLoader(augmented_train_ds, batch_size=CFG['batch_size'], 
                         shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], 
                       shuffle=False, num_workers=2, pin_memory=True)

try:
    (
        trained_model,
        bb_scores,
        wb_scores,
        history,
        initial_canary_losses,
        epsilon_spent,
    ) = train_dp_dro_with_audit_simplified(
        cfg                  = CFG,
        train_loader         = train_loader,
        val_loader           = val_loader,
        canary_ds            = canary_ds,
        membership           = membership,
        in_canary_global_idxs = in_canary_global_idxs,
    )
    
    print(f'\n✓ Training completed successfully!')
    print(f'DP-SGD ε spent: {epsilon_spent:.4f}')
    print(f'Final Male MAE: {history["val_male_mae"][-1]:.3f}')
    print(f'Final Female MAE: {history["val_female_mae"][-1]:.3f}')
    
    # Save results
    fresh_results = {
        'timestamp': datetime.now().isoformat(),
        'config': {k: str(v) for k, v in CFG.items()},
        'fairness': {
            'final_male_mae': history['val_male_mae'][-1],
            'final_female_mae': history['val_female_mae'][-1],
            'gender_gap': abs(history['val_male_mae'][-1] - history['val_female_mae'][-1]),
            'history': history,
        },
        'privacy': {
            'theoretical_epsilon': epsilon_spent,
            'delta': CFG['dp_delta'],
            'k': CFG['audit_k'],
            'audit_gamma': CFG['audit_gamma'],
        },
        'bb_scores': bb_scores.tolist(),
        'wb_scores': wb_scores.tolist(),
        'membership': membership.numpy().tolist(),
    }
    
    save_results(fresh_results, CFG, 'full')
    
except Exception as e:
    print(f'\n✗ Training failed: {e}')
    import traceback
    traceback.print_exc()
    
    # Try to load existing results
    existing = load_latest_results(CFG, 'full')
    if existing:
        bb_scores = np.array(existing['bb_scores'])
        wb_scores = np.array(existing['wb_scores'])
        membership = torch.tensor(existing['membership'])
        history = existing.get('fairness', {}).get('history', {'val_male_mae': [10], 'val_female_mae': [12]})
        epsilon_spent = existing['privacy']['theoretical_epsilon']
    else:
        raise

# ============================================================
# RUN PRIVACY AUDIT
# ============================================================

print('\n' + '='*60)
print('Running privacy audit...')
print('='*60)

bb_result = audit_epsilon(bb_scores, membership, gamma=CFG['audit_gamma'])
wb_result = audit_epsilon(wb_scores, membership, gamma=CFG['audit_gamma'])

print('\n' + '='*60)
print('PRIVACY AUDIT RESULTS (Steinke et al. 2023, Corollary 5.4)')
print('='*60)
print(f"  Theoretical ε (DP-SGD accountant) : {epsilon_spent:.4f}")
print()
print(f"  Black-box attack:")
print(f"    Correct guesses V      : {bb_result['V']} / {bb_result['k']}")
print(f"    Best threshold τ       : {bb_result['threshold']:.4f}")
print(f"    p-value (null ε=0)     : {bb_result['p_value']:.4e}")
print(f"    Empirical ε lower bound: {bb_result['eps_lb']:.4f}")
print()
print(f"  White-box attack:")
print(f"    Correct guesses V      : {wb_result['V']} / {wb_result['k']}")
print(f"    Best threshold τ       : {wb_result['threshold']:.4f}")
print(f"    p-value (null ε=0)     : {wb_result['p_value']:.4e}")
print(f"    Empirical ε lower bound: {wb_result['eps_lb']:.4f}")

# Verify scores are different
if not np.allclose(bb_scores, wb_scores, atol=1e-6):
    print('\n✓ SUCCESS: White-box scores are different from black-box scores!')
    print(f'  Correlation: {np.corrcoef(bb_scores, wb_scores)[0,1]:.3f}')
else:
    print('\n⚠ WARNING: White-box scores are identical to black-box scores')

print('\n✓ Experiment completed!')


SIMPLIFIED MEMORY-EFFICIENT TRAINING

STARTING SIMPLIFIED TRAINING


---
## Privacy Audit: Steinke et al. (2023) Corollary 5.4

Given scores $s_i$ for each canary and the ground-truth membership $m_i$:

1. **Predict membership:** $\hat{m}_i = \mathbf{1}[s_i > \text{threshold}]$
2. **Count correct guesses:** $V = \sum_i \mathbf{1}[\hat{m}_i = m_i]$
3. **Hypothesis test:** Under the null hypothesis ($\varepsilon = 0$), the best an adversary can do is $V \sim \text{Binomial}(k, 0.5)$.  For a given $\varepsilon$, the success probability is $\frac{e^\varepsilon}{1 + e^\varepsilon}$.
4. **Lower bound on ε:** Binary-search for the smallest $\hat{\varepsilon}$ such that observing $V$ successes is not significant at level $\gamma$.

In [ ]:
def audit_epsilon(scores: np.ndarray,
                  membership: torch.Tensor,
                  gamma: float = 0.05,
                  n_thresholds: int = 1000) -> dict:
    """
    Steinke et al. (2023) Corollary 5.4 — one-sided binomial test.

    For each threshold τ, compute V(τ) = number of canaries for which
    sign(score - τ) == membership.  Find the threshold maximising V.
    Then binary-search for the empirical epsilon lower bound.

    Args
    ----
    scores     : (k,) float array — MI scores (higher → more likely IN)
    membership : (k,) bool tensor — True if sample was IN training set
    gamma      : significance level (typically 0.05)
    n_thresholds: grid size for threshold sweep

    Returns
    -------
    dict with keys: eps_lb, V, k, threshold, p_value
    """
    m      = membership.numpy().astype(int)   # 1 = IN, 0 = OUT
    k      = len(scores)

    # Sweep thresholds and pick the one that maximises correct guesses
    thresholds = np.linspace(scores.min() - 1e-9, scores.max() + 1e-9, n_thresholds)
    best_V, best_tau = 0, thresholds[0]
    for tau in thresholds:
        guesses = (scores > tau).astype(int)   # 1 = predict IN
        V = (guesses == m).sum()
        if V > best_V:
            best_V, best_tau = V, tau

    V = best_V

    # p-value under null (ε = 0 → p_success = 0.5)
    p_null = stats.binom.sf(V - 1, k, 0.5)   # P(X ≥ V | p=0.5)

    # Binary search for empirical ε lower bound
    # Under ε, the success probability is e^ε / (1 + e^ε)
    # We want the largest ε such that P(X ≥ V | p=e^ε/(1+e^ε)) > γ
    # i.e., the smallest ε for which the observation is significant.
    def p_val_at_eps(eps):
        p_succ = math.exp(eps) / (1 + math.exp(eps))
        return stats.binom.sf(V - 1, k, p_succ)

    lo, hi = 0.0, 20.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if p_val_at_eps(mid) > gamma:
            lo = mid   # observation still not significant → ε could be higher
        else:
            hi = mid

    # eps_lb = lo: the largest ε for which we can reject the null at level gamma
    eps_lb = lo if p_val_at_eps(lo) <= gamma else 0.0

    return {
        'eps_lb':    eps_lb,
        'V':         int(V),
        'k':         k,
        'threshold': best_tau,
        'p_value':   float(p_null),
    }


print('Audit function defined (Steinke et al. 2023, Corollary 5.4).')

In [ ]:
print('Running privacy audit...\n')

bb_result = audit_epsilon(bb_scores, membership, gamma=CFG['audit_gamma'])
wb_result = audit_epsilon(wb_scores, membership, gamma=CFG['audit_gamma'])

print('=' * 60)
print('PRIVACY AUDIT RESULTS  (Steinke et al. 2023, Corollary 5.4)')
print('=' * 60)
print(f"  Theoretical ε (DP-SGD accountant) : {epsilon_spent:.4f}")
print()
print(f"  Black-box attack:")
print(f"    Correct guesses V      : {bb_result['V']} / {bb_result['k']}")
print(f"    Best threshold τ       : {bb_result['threshold']:.4f}")
print(f"    p-value (null ε=0)     : {bb_result['p_value']:.4e}")
print(f"    Empirical ε lower bound: {bb_result['eps_lb']:.4f}")
print()
print(f"  White-box attack:")
print(f"    Correct guesses V      : {wb_result['V']} / {wb_result['k']}")
print(f"    Best threshold τ       : {wb_result['threshold']:.4f}")
print(f"    p-value (null ε=0)     : {wb_result['p_value']:.4e}")
print(f"    Empirical ε lower bound: {wb_result['eps_lb']:.4f}")
print()
print('  Note: empirical ε_lb should be ≤ theoretical ε (DP guarantee).')
print('  White-box typically gives a tighter (higher) lower bound.')

---
## Visualisation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

epochs_x = range(1, CFG['epochs'] + 1)

# ── (1) Training loss ──────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(epochs_x, history['train_loss_dro'], color='steelblue')
ax.set_title('DRO Training Loss (Hashimoto et al.)', fontsize=11)
ax.set_xlabel('Epoch'); ax.set_ylabel('Weighted L1 loss'); ax.grid(True)

# ── (2) Gender MAE gap over training ──────────────────────────────
ax = axes[0, 1]
ax.plot(epochs_x, history['val_male_mae'],   label='Male MAE',   color='royalblue')
ax.plot(epochs_x, history['val_female_mae'], label='Female MAE', color='crimson')
ax.fill_between(epochs_x,
                history['val_male_mae'],
                history['val_female_mae'],
                alpha=0.15, color='purple', label='Gender gap')
ax.set_title('Gender MAE Gap (DRO without demographics)', fontsize=11)
ax.set_xlabel('Epoch'); ax.set_ylabel('MAE (years)'); ax.legend(); ax.grid(True)

# ── (3) DP ε over training ─────────────────────────────────────────
ax = axes[0, 2]
if history['epsilon']:
    ax.plot(epochs_x, history['epsilon'], color='darkorange')
    ax.axhline(epsilon_spent, color='red', ls='--', label=f'Final ε={epsilon_spent:.2f}')
    ax.set_title('DP-SGD Privacy Budget ε Accumulation', fontsize=11)
    ax.set_xlabel('Epoch'); ax.set_ylabel('ε (δ=1e-5)'); ax.legend(); ax.grid(True)
else:
    ax.text(0.5, 0.5, 'DP disabled', ha='center', va='center', transform=ax.transAxes)

# ── (4) Black-box MI score distributions ──────────────────────────
ax = axes[1, 0]
m_np = membership.numpy()
ax.hist(bb_scores[m_np == 1],  bins=30, alpha=0.6, label='IN  canaries', color='steelblue')
ax.hist(bb_scores[m_np == 0],  bins=30, alpha=0.6, label='OUT canaries', color='salmon')
ax.axvline(bb_result['threshold'], color='black', ls='--', label=f"τ={bb_result['threshold']:.3f}")
ax.set_title(f"Black-box MI Scores  (ε_lb={bb_result['eps_lb']:.3f})", fontsize=11)
ax.set_xlabel('Score (Δ loss)'); ax.set_ylabel('Count'); ax.legend(); ax.grid(True)

# ── (5) White-box MI score distributions ──────────────────────────
ax = axes[1, 1]
ax.hist(wb_scores[m_np == 1],  bins=30, alpha=0.6, label='IN  canaries', color='steelblue')
ax.hist(wb_scores[m_np == 0],  bins=30, alpha=0.6, label='OUT canaries', color='salmon')
ax.axvline(wb_result['threshold'], color='black', ls='--', label=f"τ={wb_result['threshold']:.3f}")
ax.set_title(f"White-box MI Scores  (ε_lb={wb_result['eps_lb']:.3f})", fontsize=11)
ax.set_xlabel('Score (gradient alignment)'); ax.set_ylabel('Count'); ax.legend(); ax.grid(True)

# ── (6) Summary bar chart ─────────────────────────────────────────
ax = axes[1, 2]
bars = ['ε theoretical', 'ε BB lower bd', 'ε WB lower bd']
vals = [epsilon_spent, bb_result['eps_lb'], wb_result['eps_lb']]
colors = ['grey', 'cornflowerblue', 'tomato']
ax.bar(bars, vals, color=colors, edgecolor='black', width=0.5)
ax.set_title('Privacy Budget Summary', fontsize=11)
ax.set_ylabel('ε'); ax.grid(axis='y')
for i, v in enumerate(vals):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)

plt.suptitle(
    'DP-SGD + Hashimoto DRO + Steinke et al. Privacy Audit — UTKFace Age Regression',
    fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()

fig_path = CFG['exp_dir'] / 'dp_dro_audit_summary.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')

---
## Save Results

In [ ]:
results = {
    'timestamp':         datetime.now().isoformat(),
    'method':            'DP-SGD + Hashimoto-DRO (alpha={}) + Steinke-Audit'.format(CFG['dro_alpha']),
    'config':            {k: str(v) for k, v in CFG.items()},
    'fairness': {
        'final_male_mae':   history['val_male_mae'][-1],
        'final_female_mae': history['val_female_mae'][-1],
        'gender_gap':       abs(history['val_male_mae'][-1] - history['val_female_mae'][-1]),
    },
    'privacy': {
        'theoretical_epsilon':    epsilon_spent,
        'delta':                  CFG['dp_delta'],
        'bb_epsilon_lb':          bb_result['eps_lb'],
        'wb_epsilon_lb':          wb_result['eps_lb'],
        'bb_V':                   bb_result['V'],
        'wb_V':                   wb_result['V'],
        'k':                      k,
        'audit_gamma':            CFG['audit_gamma'],
    },
    'history':           history,
    'bb_scores':         bb_scores.tolist(),
    'wb_scores':         wb_scores.tolist(),
    'membership':        membership.numpy().tolist(),
}

out_json = CFG['exp_dir'] / 'dp_dro_audit_results.json'
with open(out_json, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved → {out_json}')

# Save model checkpoint
ckpt_path = CFG['exp_dir'] / 'dp_dro_model.pth'
# Unwrap Opacus GradSampleModule if needed
state_dict = (trained_model._module.state_dict()
              if hasattr(trained_model, '_module')
              else trained_model.state_dict())
torch.save(state_dict, ckpt_path)
print(f'Model checkpoint saved → {ckpt_path}')

---
## Summary

| Metric | Value |
|--------|-------|
| Method | DP-SGD + Hashimoto DRO |
| DRO α (χ² ball) | `dro_alpha` |
| DP noise multiplier | `dp_noise_multiplier` |
| Theoretical ε | `epsilon_spent` |
| Empirical ε (black-box) | `bb_result['eps_lb']` |
| Empirical ε (white-box) | `wb_result['eps_lb']` |
| Gender gap (MAE, years) | `final_male_mae - final_female_mae` |

### Connection between the two papers

**Hashimoto et al.** provides the fairness mechanism: by placing higher training weight on high-loss samples, the model avoids neglecting minority subgroups — *without ever seeing group labels*. The χ²-DRO objective is equivalent to minimising the worst-case loss over all distributions within a divergence ball of radius α from the empirical distribution.

**Steinke et al.** provides the privacy audit: canaries injected with random labels are trained under DP-SGD; post-training black-box (loss reduction) and white-box (gradient alignment) scores are used to run a rigorous binomial hypothesis test that yields an empirical *lower bound* on ε. If the lower bound approaches the theoretical accountant ε, the DP guarantee is tight.

**The combination** reveals a potential tension: DRO up-weights high-loss samples, which may cause the model to memorise canaries (high-loss outliers) more aggressively — potentially tightening the empirical ε lower bound even while the theoretical ε stays constant. This is an open empirical question worth investigating.